# 02 — Интерактивный эксперимент R3D-18

Запускайте сверху вниз в чистом kernel с установленными requirements.txt и requirements-ml.txt. Код обучения находится в ml/train.py; CLI использует тот же путь выполнения.

## Как интерпретировать результат
Это классификация окон вокруг **известных отметок**, а не автоматический поиск события во всём видео.
Согласованный `within_room_unique_video` split: 30 train / 10 validation; точные копии не пересекаются.
Общий кабинет, люди и неизвестные сеансы могут пересекаться. Независимость камер и сеансов не подтверждена.
Validation используется для выбора checkpoint по loss, а не как независимый финальный тест.
Первый smoke-run предсказывал нарушение для всех validation-окон (accuracy 0.5, recall 1.0).
Это проверка работоспособности, не хорошее качество модели и не законченный детектор.
Исторические метрики и provenance сохранены в `experiments/r3d18_smoke_result.md` и прежних `runs/`.

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src/overfitted_underqualified").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите из корня репозитория или notebooks/")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import importlib.util
import json
import os
import subprocess
from datetime import UTC, datetime

import pandas as pd
import yaml
from IPython.display import display

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(ROOT / "requirements-ml.txt"),
    ])

from overfitted_underqualified.ml.plots import plot_history, show_windows
from overfitted_underqualified.ml.train import (
    build_model,
    prepare_training,
    train_prepared,
)


## 1. Конфигурация

YAML задаёт seed, 16 кадров, pretrained preprocessing и заморозку backbone. Для короткой проверки задайте TRAIN_EPOCHS=1. По умолчанию применяется число эпох из YAML. Каждый запуск пишет в новый каталог; старые runs не перезаписываются. Для CPU явно задайте TRAIN_DEVICE=cpu.
По умолчанию обучение выключено; настройки читаются из .env: для нового эксперимента явно задайте RUN_TRAINING=true в корневом .env. В режиме исследования используются сохранённые результаты; число эпох из YAML применяется только при включённом обучении.


In [ ]:
from dotenv import dotenv_values

# Read afresh on every execution; shell variables take precedence over .env.
# Do not inject settings into os.environ: changing .env should take effect on rerun.
settings = {**dotenv_values(ROOT / ".env"), **os.environ}
run_training_value = str(settings.get("RUN_TRAINING", "false")).strip().lower()
if run_training_value not in {"true", "false", "1", "0", "yes", "no", "on", "off"}:
    raise ValueError("RUN_TRAINING must be true/false")
RUN_TRAINING = run_training_value in {"true", "1", "yes", "on"}
EXISTING_RUN = Path(settings.get("EXISTING_RUN") or "runs/r3d18_smoke")
if not EXISTING_RUN.is_absolute():
    EXISTING_RUN = ROOT / EXISTING_RUN


config = yaml.safe_load((ROOT / "configs/r3d18.yaml").read_text(encoding="utf-8"))
run_name = datetime.now(UTC).strftime("r3d18_notebook_%Y%m%dT%H%M%S_%fZ")
config["output_dir"] = (settings.get("TRAIN_OUTPUT_DIR") or str(ROOT / "runs" / run_name))
train_epochs = settings.get("TRAIN_EPOCHS")
if train_epochs:
    config["epochs"] = int(train_epochs)
if settings.get("TRAIN_DEVICE"):
    config["device"] = settings["TRAIN_DEVICE"]
if settings.get("VIDEO_DIR"):
    config["video_dir"] = settings["VIDEO_DIR"]

print("RUN_TRAINING:", RUN_TRAINING, "epochs:", config["epochs"], "device:", config["device"])


## 2. Данные и split

Проверяем SHA-256 реальных файлов, покрытие исходных ID, метки/отметки events.csv и отсутствие пересечения точных копий. События X-4 и X-47 сохраняются как два окна одного канонического видео. Validation не участвует в обновлении весов.

In [ ]:
inspection_config = dict(config)
if not RUN_TRAINING:
    inspection_config["device"] = "cpu"
prepared = prepare_training(inspection_config)
display(pd.crosstab(
    pd.Series([r.split for r in prepared["records"]], name="split"),
    pd.Series([r.label for r in prepared["records"]], name="label"),
))
print("Окон:", {subset: len(dataset) for subset, dataset in prepared["datasets"].items()})
print("Хеш разметки:", prepared["manifest"]["events_sha256"])
print("Хеш split:", prepared["manifest"]["split_sha256"])

window_table = pd.DataFrame([
    {"split": subset, "clip_id": record.clip_id, "label": record.label,
     "center_sec": center, "source_ids": list(record.aliases), "last_frame_sec": record.last_time}
    for subset, dataset in prepared["datasets"].items() for record, center in dataset.samples
])
display(window_table)
display(pd.crosstab(window_table["split"], window_table["label"]))


## 3. Примеры окон

Показаны четыре из 16 выбранных кадров до crop/нормализации. Проверим итоговый тензор pretrained-преобразования.

In [ ]:
train_data = prepared["datasets"]["train"]
example_indices = [next(i for i, (record, _) in enumerate(train_data.samples) if record.label == label) for label in (0, 1)]
show_windows(train_data, example_indices)
print("После preprocessing:", tuple(train_data[0]["video"].shape), "= C,T,H,W")


## 4. Загрузка R3D-18

Pretrained-веса берутся из кэша, при первом запуске скачиваются. Бинарная голова обучается; при freeze_backbone=true параметры backbone и статистики BatchNorm фиксированы.

In [ ]:
model = None
if RUN_TRAINING:
    model = build_model(prepared["config"]["weights"], prepared["config"]["freeze_backbone"])
    print(model.fc)
    print("Обучаемых параметров:", sum(p.numel() for p in model.parameters() if p.requires_grad))
else:
    print("R3D-18: pretrained backbone → Linear(512, 2). Загрузка весов пропущена.")


## 5. Обучение

Используем те же DataLoader, seed и цикл эпох, что CLI. Validation выполняется в eval mode без градиентов. Повторное выполнение этой ячейки с тем же output_dir отклоняется.

In [ ]:
if RUN_TRAINING:
    output_dir = train_prepared(prepared, model)
else:
    output_dir = EXISTING_RUN
    print("Обучение выключено. Анализируем сохранённый запуск:", output_dir)


## 6. Loss и артефакты

Для одного smoke-epoch график содержит по одной точке: по нему нельзя судить о сходимости.

In [ ]:
history = []
if (output_dir / "history.json").is_file():
    history = json.loads((output_dir / "history.json").read_text(encoding="utf-8"))
    saved_config = yaml.safe_load((output_dir / "config.yaml").read_text(encoding="utf-8"))
    display(saved_config)  # Результаты относятся к этой конфигурации, не к новым настройкам выше.
    plot_history(history)
    display(pd.json_normalize(history))
    display(pd.DataFrame({"artifact": [p.name for p in sorted(output_dir.iterdir())]}))
    print("Checkpoint для 03_evaluation:", output_dir / "best.pt")
else:
    print("Сохранённого запуска нет. Укажите EXISTING_RUN либо явно включите обучение.")


## 7. Learning curves и validation выбранного checkpoint
Сопоставляем loss и accuracy по эпохам. Валидационные метрики берём из сохранённой истории;
реализация метрик остаётся в модуле. Лучший checkpoint выбран по validation loss, а не по последней эпохе.

In [ ]:
import matplotlib.pyplot as plt

from overfitted_underqualified.ml.plots import plot_confusion

if history:
    curves = pd.json_normalize(history).set_index("epoch")
    curves[["train.accuracy", "validation.accuracy"]].plot(marker="o", ylabel="Accuracy", ylim=(0, 1))
    plt.show()
    plt.close("all")
    best_epoch = min(history, key=lambda epoch: epoch["validation"]["loss"])
    display(pd.Series(best_epoch["validation"], name=f"best epoch {best_epoch['epoch']}"))
    plot_confusion(best_epoch["validation"])


## 8. Сравнение сохранённых экспериментов
Добавляйте каталоги вручную. Сравниваем только запуски с одинаковыми входами и окнами.
Иная длительность окна или выборка требует отдельного протокола; не смешиваем такие метрики.
Повторы одного smoke-run проверяют воспроизводимость, а не дают независимые оценки качества.

In [ ]:
RUNS_TO_COMPARE = [ROOT / "runs/r3d18_smoke", ROOT / "runs/r3d18_smoke_repeat", output_dir]
comparison = []
reference_manifest = None
for run_dir in dict.fromkeys(RUNS_TO_COMPARE):
    if not all((run_dir / name).is_file() for name in ("history.json", "data_manifest.json", "config.yaml")):
        print("Пропущен незавершённый/отсутствующий запуск:", run_dir)
        continue
    manifest = json.loads((run_dir / "data_manifest.json").read_text(encoding="utf-8"))
    run_config = yaml.safe_load((run_dir / "config.yaml").read_text(encoding="utf-8"))
    signature = {key: manifest[key] for key in ("events_sha256", "split_sha256", "windows")}
    signature["sampling"] = {key: run_config[key] for key in ("frames", "window_sec", "negative_windows", "weights")}
    if reference_manifest is None:
        reference_manifest = signature
    if signature != reference_manifest:
        print("Другой набор данных/окон, не сравниваем напрямую:", run_dir)
        continue
    run_history = json.loads((run_dir / "history.json").read_text(encoding="utf-8"))
    run_config = yaml.safe_load((run_dir / "config.yaml").read_text(encoding="utf-8"))
    best = min(run_history, key=lambda epoch: epoch["validation"]["loss"])
    comparison.append({"run": str(run_dir.relative_to(ROOT)) if run_dir.is_relative_to(ROOT) else str(run_dir),
                       "epoch": best["epoch"], "seed": run_config["seed"],
                       "learning_rate": run_config["learning_rate"],
                       **{key: best["validation"][key] for key in ("loss", "accuracy", "precision", "recall", "f1")}})
display(pd.DataFrame(comparison))


## 9. Выводы эксперимента
Для первого smoke-run высокая полнота сочеталась с предсказанием violation для каждого окна:
accuracy=0.5 не превосходит тривиальное постоянное предсказание на сбалансированной validation.
Одна эпоха подтверждает выполнение pipeline, но не сходимость.

Для следующего эксперимента запишите гипотезу, одно изменяемое условие, seed и новый output_dir.
После запуска дополните вывод: динамика train/validation, типичные ошибки, подтверждена ли гипотеза.
Не подбирайте порог в этом отчёте. Разбор отдельных ошибок — в 03_evaluation.
Перед коммитом очищайте outputs: таблицы и кадры могут содержать банковские данные.